# Fase 2 · Transformación de datos en los datasets

## Objetivo

El objetivo de esta fase consiste en ejecutar los cambios y ajustes detectados durante el Análisis Exploratorio de Datos (EDA), para unificar, limpiar y transformar los datasets seleccionados en este proyecto.

En esta etapa se trabaja principalmente en:

- abordar los duplicados,
- combinar varios datasets,
- gestionar los valores nulos,
- homogenizar categorías para asegurar la integridad semántica,
- y exportar los archivos finales ya depurados.

Este proceso permitirá tener un conjunto de datos más consistentes, comparables y listos para la siguiente fase de análisis avanzado y visualización.

In [2]:
# Importación de librerías
import pandas as pd
import numpy as np
import re

# Configuración del sistema para encontrar la carpeta raíz
import sys
import os

# Esto obliga a Python a mirar una carpeta hacia atrás (donde está 'src')
sys.path.append(os.path.abspath(os.path.join('..')))
sys.path.append(os.path.abspath(os.path.join('.')))

# Importación de módulos de transformación 
from src.etl.load_data import load_friends_data_raw, load_friends_data_translated
from src.etl import transform as trans
from transformers import pipeline
from src.etl import column_standardizer as stan


 
# Configuración para visualizar todas las columnas del DataFrame
pd.set_option('display.max_columns', None) 

In [3]:
dfs = load_friends_data_raw()

[2026-05-28 10:01:58] INFO - Cargando datasets desde: C:\Users\dacil\Desktop\Adalab\pair\modulo_4_pair\friends-analytics-workflow\data_raw
[2026-05-28 10:01:58] INFO - → Cargando weddings_divorces_ross.csv...
[2026-05-28 10:01:58] INFO - → Cargando friends_cameos.csv...
[2026-05-28 10:01:58] INFO - → Cargando friends_emotions.csv...
[2026-05-28 10:01:59] INFO - → Cargando friends_episodes.csv...
[2026-05-28 10:01:59] INFO - → Cargando friends_sets.csv...
[2026-05-28 10:01:59] INFO - → Cargando friends_info.csv...
[2026-05-28 10:01:59] INFO - → Cargando friends_quotes.csv...
[2026-05-28 10:02:00] INFO - → Cargando friends.csv...
[2026-05-28 10:02:00] INFO - → Cargando phoebe_buffay_songs.csv...
[2026-05-28 10:02:00] INFO - → Cargando duck_and_chicken.csv...
[2026-05-28 10:02:00] INFO - Todos los datasets fueron cargados correctamente.


## 1. Transformación de  las variables numéricas (friends_quotes) de orden de float a entero (int) para mejorar la estructura. 

In [4]:
df_quotes = dfs["quotes"]

df_quotes.head()

,author,episode_number,episode_title,quote,quote_order,season
0,Monica,1.0,Monica Gets A Roommate,There's nothing to tell! He's just some guy I ...,0.0,1.0
1,Joey,1.0,Monica Gets A Roommate,"C'mon, you're going out with the guy! There's ...",1.0,1.0
2,Chandler,1.0,Monica Gets A Roommate,"All right Joey, be nice. So does he have a hum...",2.0,1.0
3,Phoebe,1.0,Monica Gets A Roommate,"Wait, does he eat chalk?",3.0,1.0
4,Phoebe,1.0,Monica Gets A Roommate,"Just, 'cause, I don't want her to go through w...",4.0,1.0


In [5]:
df_quotes["quote_order"] = df_quotes["quote_order"].astype(int)
df_quotes["season"] = df_quotes["season"].astype(int)
df_quotes["episode_number"] = df_quotes["episode_number"].astype(int)


In [6]:
df_quotes.head()

,author,episode_number,episode_title,quote,quote_order,season
0,Monica,1,Monica Gets A Roommate,There's nothing to tell! He's just some guy I ...,0,1
1,Joey,1,Monica Gets A Roommate,"C'mon, you're going out with the guy! There's ...",1,1
2,Chandler,1,Monica Gets A Roommate,"All right Joey, be nice. So does he have a hum...",2,1
3,Phoebe,1,Monica Gets A Roommate,"Wait, does he eat chalk?",3,1
4,Phoebe,1,Monica Gets A Roommate,"Just, 'cause, I don't want her to go through w...",4,1


In [7]:
df_quotes.to_csv("../data_processed/friends_quotes.csv", index=False, encoding="utf-8")

## 2. Limpiar y estandarizar la columna written_by (friends_info)

In [8]:
df_info= dfs["info"]

df_info.head()

,season,episode,title,directed_by,written_by,air_date,us_views_millions,imdb_rating
0,1,1,The Pilot,James Burrows,David Crane & Marta Kauffman,1994-09-22,21.5,8.3
1,1,2,The One with the Sonogram at the End,James Burrows,David Crane & Marta Kauffman,1994-09-29,20.2,8.1
2,1,3,The One with the Thumb,James Burrows,Jeffrey Astrof & Mike Sikowitz,1994-10-06,19.5,8.2
3,1,4,The One with George Stephanopoulos,James Burrows,Alexa Junge,1994-10-13,19.7,8.1
4,1,5,The One with the East German Laundry Detergent,Pamela Fryman,Jeff Greenstein & Jeff Strauss,1994-10-20,18.6,8.5


In [9]:
trans.process_friends_writers(df_info)

¡Fichero corregido con éxito! Guardado en: C:\Users\dacil\Desktop\Adalab\pair\modulo_4_pair\friends-analytics-workflow\data_processed\writers.csv (303 filas).


,season,episode,writer
0,1,1,David Crane
1,1,1,Marta Kauffman
2,1,2,David Crane
3,1,2,Marta Kauffman
4,1,3,Jeffrey Astrof
...,...,...,...
298,10,16,Ted Cohen
299,10,17,Marta Kauffman
300,10,17,David Crane
301,10,18,Marta Kauffman


In [10]:
df_info.drop("written_by", axis=1, inplace=True)


df_info.head(2)

,season,episode,title,directed_by,air_date,us_views_millions,imdb_rating
0,1,1,The Pilot,James Burrows,1994-09-22,21.5,8.3
1,1,2,The One with the Sonogram at the End,James Burrows,1994-09-29,20.2,8.1


In [11]:
df_info.to_csv("../data_processed/friends_info.csv", index=False, encoding="utf-8") 

#### Traducir las columnas

In [12]:
df_dac = dfs["dac"]

In [13]:
df_dac = stan.standardize_columns(df_dac)

In [14]:
df_dac.head()

,season,episode_number,animal,accion
0,3,3x21,Pollito,Joey lo compra
1,3,3x22,Pollito,Joey y Chandler cuidan de él.
2,3,3x22,Pato,Chandler lo rescata para que el pollito tenga ...
3,3,3x25,Pollito,Aparecen en el apartamento de los chicos.
4,3,3x25,Pato,Aparecen en el apartamento de los chicos.


In [15]:
df_dac.to_csv("../data_processed/duck_and_chicken.csv", index=False, encoding="utf-8")

In [16]:
df_cameos = dfs["cameos"]
df_cameos.head(1)

,Actor/Actriz,Personaje,Descripción/Temporada
0,Brad Pitt,Will Colbert,Antiguo compañero que odiaba a Rachel (T8)


In [17]:
# 1. Extraemos la descripción y el número de la temporada usando Regex
# El patrón busca "T" seguido de uno o más números d+ dentro de un paréntesis
df_extracted = df_cameos["Descripción/Temporada"].str.extract(r"(?P<descripcion>.*?)\s*\(T(?P<temporada>\d+)\)")

# 2. Asignamos los resultados de vuelta a nuestro DataFrame original
df_cameos["descripcion"] = df_extracted["descripcion"]
df_cameos["temporada"] = df_extracted["temporada"]

# 3. Borramos la columna vieja que ya no necesitamos
df_cameos = df_cameos.drop(columns=["Descripción/Temporada"])

# Ver el resultado
df_cameos.head(1)

,Actor/Actriz,Personaje,descripcion,temporada
0,Brad Pitt,Will Colbert,Antiguo compañero que odiaba a Rachel,8


In [18]:
df_cameos = stan.standardize_columns(df_cameos)

In [19]:
df_cameos.head()

,actor,author,description,season
0,Brad Pitt,Will Colbert,Antiguo compañero que odiaba a Rachel,8
1,Bruce Willis,Paul Stevens,Padre de Elizabeth y novio de Rachel,6
2,Julia Roberts,Susie Moss,Compañera de primaria de Chandler,2
3,Charlie Sheen,Ryan,Marinero novio de Phoebe que tiene varicela,2
4,Danny DeVito,Roy,El stripper sensible en la despedida de Phoebe,10


In [20]:
df_cameos.to_csv("../data_processed/friends_cameos.csv", index=False, encoding="utf-8")

In [21]:
df_sets = dfs["sets"]

In [22]:
df_sets = stan.standardize_columns(df_sets)

# tenemos que cambiar las columnas del fichero resultante final.
# lo ideal sería que los dos .py de traducción fueran genéricos y no tuvieran que ser modificados para cada proyecto, sino que se les pasara el nombre del csv a traducir, las columnas a traducir, etc.
# revisar fichero quotes translated.

In [23]:
dfs_finales = load_friends_data_translated()

[2026-05-28 10:02:02] INFO - Cargando datasets desde: C:\Users\dacil\Desktop\Adalab\pair\modulo_4_pair\friends-analytics-workflow\data_translated
[2026-05-28 10:02:02] INFO - → Cargando friends_weddings_divorce_ross.csv...
[2026-05-28 10:02:02] INFO - → Cargando friends_cameos.csv...
[2026-05-28 10:02:02] INFO - → Cargando friends_emotions.csv...
[2026-05-28 10:02:02] INFO - → Cargando friends_episodes.csv...
[2026-05-28 10:02:02] INFO - → Cargando friends_sets.csv...
[2026-05-28 10:02:02] INFO - → Cargando friends_info.csv...
[2026-05-28 10:02:02] INFO - → Cargando friends_quotes.csv...
[2026-05-28 10:02:03] INFO - → Cargando friends.csv...
[2026-05-28 10:02:04] INFO - → Cargando friends_songs.csv...
[2026-05-28 10:02:04] INFO - → Cargando duck_and_chicken.csv...
[2026-05-28 10:02:04] INFO - → Cargando writers.csv...
[2026-05-28 10:02:04] INFO - Todos los datasets fueron cargados correctamente.


In [24]:
df_quotes = dfs_finales["quotes"]

In [25]:
df_quotes["autor"] = df_quotes["autor"].str.title()

In [26]:
df_quotes.sample(20)

,autor,numero_episodio,titulo_episodio,cita,orden_cita,temporada
40470,Joey,16,La verdad sobre Londres,"¡Sí, cariño!",201,7
6531,Janice,3,Muere Heckles,"Oye, son todos.",131,2
31763,Chandler,8,Los dientes de Ross,"Ya sabes, Dios mío.",94,6
16166,Kate,19,La camiseta diminuta,¿De dónde te conozco?,41,3
11449,Rachel,24,La boda de Barry y Mindy,"Dios, lo sé, tienes razón.",115,2
4533,Rachel,19,El mono se escapa,"Oh. Oh, esas pequeñas y toscas cosas Amish que...",60,1
57773,Ross,10,Chandler es atrapado,"¡Ese dinero es mío, Green!",143,10
15629,Chandler,17,El viaje de esquí,"Sí, sí, es sólo que nosotros, ah, ya estábamos...",39,3
23529,Chandler,22,El peor padrino de todos los tiempos,Gracias hombre.,251,4
2861,Phoebe,12,La Docena de Lasañas,¡Ey!,148,1


In [27]:
# ver fila 50805 <Joey Looks>\r\n Oh! Stupid Long Sleeves.	
# 50969, 50980, 50989

#### Utilizar una funcion para detectar las lineas que no son diálogo.

In [28]:
trans.generar_log_etiquetas(path_in="../data_translated/friends_quotes.csv",path_log="friends_quotes_clean.log", columns=["autor", "cita"])

True

In [29]:
trans.generar_archivo_limpio(path_in="../data_translate/friends_quotes_clean.csv", path_log="friends_quotes_clean.log",
    path_out="../data_translate/friends_quotes_clean_v2.csv")

ValueError: El archivo .log no contiene filas válidas para eliminar.